# Conversion Rate -- Prediction du taux de conversion

**Problematique** : Comment predire si un visiteur va convertir (achat, inscription) a partir de ses caracteristiques demographiques et comportementales ?

**Approche** :
1. EDA et nettoyage des donnees (valeurs aberrantes)
2. Analyse du desequilibre des classes
3. Modelisation : Regression Logistique, Random Forest, XGBoost
4. Optimisation des hyperparametres par GridSearchCV
5. Feature Importance et recommandations business

**Metrique principale** : F1-Score (classe 1) en raison du desequilibre des classes.

## 1. Analyse exploratoire et preparation des donnees

Importation des librairies et chargement du dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import (
    train_test_split, GridSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# Configuration de l'affichage
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
warnings.filterwarnings("ignore")

### 1.1 Chargement du dataset

Chargement du fichier d'entrainement depuis le dossier `data/raw`.

In [ ]:
# Chargement du dataset d'entrainement
df = pd.read_csv("../data/raw/conversion_data_train.csv")

print(
    f"Dimensions du dataset : {df.shape[0]} lignes, "
    f"{df.shape[1]} colonnes."
)
display(df.head())

### 1.2 Audit des donnees

Verification des types, valeurs nulles et anomalies.

In [ ]:
# Verification des types et des valeurs nulles
df.info()

print("-" * 30)
print("Pourcentage de valeurs manquantes par colonne :")
print(df.isnull().mean() * 100)

In [ ]:
# Statistiques descriptives
# Detection des anomalies (ex : age negatif ou > 100)
df.describe()

### 1.3 Analyse de la variable cible

Distribution de `converted`. Le desequilibre des classes impose d'utiliser le F1-Score plutot que l'Accuracy.

In [ ]:
# Distribution de la variable cible
conversion_rate = (
    df["converted"].value_counts(normalize=True) * 100
)

print(
    f"Taux de non-conversion (0) : "
    f"{conversion_rate[0]:.2f}%"
)
print(
    f"Taux de conversion (1)     : "
    f"{conversion_rate[1]:.2f}%"
)

plt.figure(figsize=(6, 4))
sns.countplot(
    x="converted", hue="converted",
    legend=False, data=df, palette="viridis"
)
plt.title(
    "Distribution de la variable cible 'Converted'"
)
plt.show()

### 1.4 Analyse des correlations et comportements

Taux de conversion par pays et par source de trafic.

In [ ]:
# Taux de conversion par pays et par source
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(
    x="country", y="converted", hue="country",
    legend=False, data=df, ax=axes[0], palette="magma"
)
axes[0].set_title("Taux de conversion par pays")

sns.barplot(
    x="source", y="converted", hue="source",
    legend=False, data=df, ax=axes[1], palette="magma"
)
axes[1].set_title(
    "Taux de conversion par source de trafic"
)

plt.show()

**Observations** :
- Disparites de conversion selon les pays (China et US plus eleves).
- La source de trafic a un impact limite sur le taux de conversion.

Distribution de l'age et impact du nombre de pages visitees sur la conversion.

In [ ]:
# Distribution des ages selon la conversion
plt.figure(figsize=(12, 5))
sns.histplot(
    data=df, x="age", hue="converted",
    multiple="stack", bins=30, palette="viridis"
)
plt.title("Distribution des ages selon la conversion")
plt.show()

# Impact du nombre de pages visitees
plt.figure(figsize=(12, 5))
sns.boxplot(
    x="converted", y="total_pages_visited",
    hue="converted", legend=False,
    data=df, palette="viridis"
)
plt.title(
    "Impact du nombre de pages vues sur la conversion"
)
plt.show()

### 1.5 Nettoyage des donnees

Suppression des valeurs aberrantes : age max = 123 ans (probablement des erreurs de saisie).

In [ ]:
# Suppression des ages aberrants (> 100 ans)
print(f"Lignes avant nettoyage : {len(df)}")
df = df[df["age"] < 100]
print(f"Lignes apres nettoyage : {len(df)}")

### 1.6 Synthese de l'EDA

**Observations principales** :
- Le dataset est fortement desequilibre : environ 97 % de non-conversions vs 3 % de conversions. L'Accuracy serait trompeuse, le F1-Score est la metrique adaptee.
- Le nombre de pages visitees (`total_pages_visited`) est le predicteur le plus discriminant : les visiteurs qui consultent plus de 12-15 pages convertissent quasi systematiquement.
- Les jeunes (18-30 ans) convertissent legerement mieux que les tranches plus agees.
- Les disparites geographiques existent mais restent moderees.
- Aucune valeur manquante, seulement 2 lignes avec un age aberrant (> 100 ans) supprimees.

## 2. Modelisation

### 2.1 Preprocessing

- One-Hot Encoding des variables categorielles (`country`, `source`).
- Split Train/Test stratifie 80/20.
- StandardScaler pour la normalisation.

In [ ]:
# Separation target / features
target_name = "converted"
X = df.drop(target_name, axis=1)
y = df[target_name]

# One-Hot Encoding des variables categorielles
X = pd.get_dummies(X, drop_first=True)

# Split train/test stratifie (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation (fit sur train uniquement)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(
    f"Train : {X_train_scaled.shape}, "
    f"Test : {X_test_scaled.shape}"
)

### 2.2 Baseline : Regression Logistique

Premier modele simple avec `class_weight='balanced'` pour compenser le desequilibre des classes.

In [ ]:
# Regression Logistique (baseline)
model_lr = LogisticRegression(
    class_weight="balanced", random_state=42
)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)

print("Classification Report (Baseline) :")
print(classification_report(y_test, y_pred_lr))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, ax=ax, cmap="Blues"
)
ax.set_title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()

print(
    f"\nF1-Score (classe 1) : "
    f"{f1_score(y_test, y_pred_lr):.4f}"
)

**Interpretation** : La regression logistique avec ponderation des classes obtient un F1-Score d'environ 0.51 pour la classe 1. C'est un point de depart acceptable mais insuffisant : le modele lineaire ne capture pas bien les interactions non lineaires entre les features. Les modeles ensemblistes devraient ameliorer significativement ce resultat.

### 2.3 Random Forest

Ensemble d'arbres de decision par bagging. `class_weight='balanced'` pour gerer le desequilibre.

In [ ]:
# Random Forest avec ponderation des classes
model_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model_rf.fit(X_train_scaled, y_train)

y_pred_rf = model_rf.predict(X_test_scaled)

print("Classification Report (Random Forest) :")
print(classification_report(y_test, y_pred_rf))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, ax=ax, cmap="Oranges"
)
ax.set_title("Confusion Matrix - Random Forest")
plt.tight_layout()
plt.show()

print(
    f"\nF1-Score (classe 1) : "
    f"{f1_score(y_test, y_pred_rf):.4f}"
)

**Interpretation** : Le Random Forest ameliore le F1-Score a environ 0.58, un gain notable par rapport a la baseline. Cependant, le modele reste en dessous de 0.60, ce qui laisse penser qu'un modele de boosting pourrait mieux capturer les patterns de conversion.

### 2.4 XGBoost

Gradient boosting, particulierement performant sur les donnees tabulaires.

In [ ]:
# XGBoost avec parametres par defaut
model_xgb = XGBClassifier(random_state=42)
model_xgb.fit(X_train_scaled, y_train)

y_pred_xgb = model_xgb.predict(X_test_scaled)

print("Classification Report (XGBoost) :")
print(classification_report(y_test, y_pred_xgb))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb, ax=ax, cmap="Greens"
)
ax.set_title("Confusion Matrix - XGBoost")
plt.tight_layout()
plt.show()

print(
    f"\nF1-Score (classe 1) : "
    f"{f1_score(y_test, y_pred_xgb):.4f}"
)

**Interpretation** : XGBoost avec les parametres par defaut atteint un F1-Score d'environ 0.75, une amelioration considerable par rapport aux modeles precedents. Le gradient boosting capture mieux les interactions non lineaires entre les features. L'optimisation des hyperparametres devrait encore ameliorer ces resultats.

### 2.5 Optimisation des hyperparametres (GridSearchCV)

Recherche exhaustive sur une grille de parametres, optimisant le F1-Score en cross-validation 3-fold.

In [ ]:
# Grille de parametres pour XGBoost
param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
    "n_estimators": [100, 200],
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    scoring="f1",
    cv=3,
    verbose=1,
    n_jobs=-1,
)

grid_search.fit(X_train_scaled, y_train)

print(
    f"\nMeilleurs hyperparametres : "
    f"{grid_search.best_params_}"
)
print(
    f"Meilleur F1-Score (CV) : "
    f"{grid_search.best_score_:.4f}"
)

# Evaluation du meilleur modele sur le test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

print(
    "\nClassification Report (XGBoost Optimise) :"
)
print(classification_report(y_test, y_pred_best))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best, ax=ax, cmap="RdYlGn"
)
ax.set_title(
    "Confusion Matrix - XGBoost Optimise (GridSearchCV)"
)
plt.tight_layout()
plt.show()

print(
    f"\nF1-Score (classe 1) : "
    f"{f1_score(y_test, y_pred_best):.4f}"
)

### 2.6 Comparaison des modeles (Cross-Validation)

Evaluation robuste de tous les modeles en cross-validation 3-fold pour comparer les performances.

In [ ]:
# Cross-validation F1 pour chaque modele (cv=3)
models = {
    "Logistic Regression": model_lr,
    "Random Forest": model_rf,
    "XGBoost (Defaut)": model_xgb,
    "XGBoost (Optimise)": best_model,
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(
        model, X_train_scaled, y_train,
        cv=3, scoring="f1", n_jobs=-1,
    )
    test_f1 = f1_score(
        y_test, model.predict(X_test_scaled)
    )
    results.append({
        "Modele": name,
        "F1 CV (Mean)": round(cv_scores.mean(), 4),
        "F1 CV (Std)": round(cv_scores.std(), 4),
        "F1 Test": round(test_f1, 4),
    })

comparison_df = pd.DataFrame(results)
print("Comparaison des modeles :")
display(comparison_df)

## 3. Feature Importance

Variables les plus determinantes selon le meilleur modele (XGBoost optimise).

In [ ]:
# Feature importance du meilleur modele
feature_names = X.columns
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(
    x=importances[indices],
    y=feature_names[indices],
    hue=feature_names[indices],
    legend=False,
    palette="viridis",
)
plt.title("Feature Importance (XGBoost Optimise)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 4. Prediction sur les donnees de test (Submission)

Application du meilleur modele sur `conversion_data_test.csv` avec le meme preprocessing.

In [ ]:
# Chargement des donnees de test
df_final = pd.read_csv(
    "../data/raw/conversion_data_test.csv"
)
print(f"Donnees finales chargees : {df_final.shape}")

# Preprocessing identique a l'entrainement
X_final = pd.get_dummies(df_final, drop_first=True)

# Alignement des colonnes avec le jeu d'entrainement
features_train = X.columns
X_final = X_final.reindex(
    columns=features_train, fill_value=0
)

# Normalisation avec le scaler deja entraine
X_final_scaled = scaler.transform(X_final)

# Predictions avec le meilleur modele
y_final_pred = best_model.predict(X_final_scaled)

# Generation du fichier de soumission
submission = pd.DataFrame({"converted": y_final_pred})
submission.to_csv(
    "../data/processed/submission.csv", index=False
)

print("Fichier 'submission.csv' genere.")
print("Distribution des predictions :")
print(pd.Series(y_final_pred).value_counts())

## 5. Conclusion et recommandations

### Bilan des performances

| Modele | F1 (CV) | F1 (Test) |
|--------|---------|-----------|
| Logistic Regression | 0.5111 | 0.5118 |
| Random Forest | 0.5754 | 0.5761 |
| XGBoost (defaut) | 0.7561 | 0.7544 |
| **XGBoost (optimise)** | **0.7650** | **0.7591** |

### Leviers d'action

1. **total_pages_visited** : predicteur dominant. Au-dela de 12-15 pages, conversion quasi-certaine. Inciter la navigation (liens internes, suggestions).
2. **age** : les 18-30 ans convertissent mieux. Cibler les campagnes sur cette tranche.
3. **new_user** : les utilisateurs recurrents convertissent mieux. Mettre en place du retargeting.
4. **country** : disparites geographiques. Adapter le contenu par region.

### Limites

- Le desequilibre des classes (97/3) reste un defi majeur.
- La grille de parametres pour GridSearchCV pourrait etre elargie.
- D'autres modeles (LightGBM, stacking) pourraient etre testes pour ameliorer le F1-Score.